In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import os

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = f'{os.environ["JAVA_HOME"]}/bin:' + os.environ["PATH"]

spark = SparkSession.builder\
    .appName("wikiChanges")\
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/08 12:38:22 WARN Utils: Your hostname, Mishkas-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.4.72 instead (on interface en0)
25/08/08 12:38:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/08 12:38:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
raw_df = spark.read.json("/Users/mish/Documents/Code/Projects/wikiEmbeddings/data/20000.json")

clean_df = raw_df.select(
    col("meta.dt").alias("timestamp"),
    "user",
    "type",
    "title",
    "bot",
    "wiki",
    "comment",
    col("meta.uri").alias("uri"),
    col("meta.domain").alias("domain")
)

In [15]:
clean_df.head(1)
import pandas as pd
c = clean_df.toPandas()
c.head(5)
# c['timestamp'].dtype

,timestamp,user,type,title,bot,wiki,comment,uri,domain
0,2025-08-08T17:13:58.666Z,Alexkrakovsky,categorize,Category:All media needing categories as of 2025,False,commonswiki,[[:File:ДАХмО 685-3-97. 1843. Метрична книга к...,https://commons.wikimedia.org/wiki/Category:Al...,commons.wikimedia.org
1,2025-08-08T17:13:58.732Z,SchlurcherBot,edit,"File:Stralsund, Alter Markt 11 (2008-05-05).JPG",True,commonswiki,/* wbeditentity-update:0| */ automatically add...,https://commons.wikimedia.org/wiki/File:Strals...,commons.wikimedia.org
2,2025-08-08T17:13:58.747Z,ListeriaBot,edit,Gebruiker:Jane023/schilderijen van Jan Steen,False,nlwiki,Wikidata list updated [V2],https://nl.wikipedia.org/wiki/Gebruiker:Jane02...,nl.wikipedia.org
3,2025-08-08T17:13:58.913Z,Ideophagous,edit,Q8408430,False,wikidatawiki,/* wbsetdescription-add:1|ary */ تصنيف د ويكيم...,https://www.wikidata.org/wiki/Q8408430,www.wikidata.org
4,2025-08-08T17:13:58.900Z,2.207.102.157,log,Template:nl-decl-indefinite article,False,enwiktionary,,https://en.wiktionary.org/wiki/Template:nl-dec...,en.wiktionary.org


In [7]:
clean_df.write \
    .mode("overwrite") \
    .partitionBy("timestamp") \
    .parquet("s3://wikichangesbucket/processed/")


Py4JJavaError: An error occurred while calling o62.parquet.
: org.apache.hadoop.fs.UnsupportedFileSystemException: No FileSystem for scheme "s3"
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3581)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3612)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:172)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3716)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3667)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:557)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:366)
	at org.apache.spark.sql.execution.datasources.DataSource.makeQualified(DataSource.scala:125)
	at org.apache.spark.sql.execution.datasources.DataSource.planForWritingFileFormat(DataSource.scala:468)
	at org.apache.spark.sql.execution.datasources.DataSource.planForWriting(DataSource.scala:554)
	at org.apache.spark.sql.classic.DataFrameWriter.saveToV1Source(DataFrameWrit